# Smoke Test — 1-2분 안에 전체 파이프라인 검증

Phase 6 전체 (~50분) 돌리기 전에 데이터 attach, 모델 로드, normalization, regex seeds, anchors, submission 형식이 다 작동하는지 빠르게 확인.

## 검증 항목

| Cell | 검증 | 통과 기준 |
|---|---|---|
| 1 | 데이터 경로 attach | 6개 파일 다 `ok` |
| 2 | normalize module | train 97%+ resolve |
| 3 | BGE-M3 모델 로드 (encode 안 함) | "BGE-M3 loaded" |
| 4 | BM25 laws 인덱싱 (caching) | "BM25 ready" |
| 5 | 토픽 분류기 + anchors | val 10개 토픽 분류 출력 |
| 6 | predict (seeds + anchors + BM25 만) | 5개 val 결과 출력 |
| 7 | 전체 val Macro F1 | 0.05~0.15 사이 (작은 베이스라인) |
| 8 | submission.csv 생성 | rows=40, corpus coverage=1.0 |

## 실행 시간

- CPU 만 (GPU 없어도 됨): ~1분
- GPU T4×1 또는 ×2: ~1~2분 (BGE-M3 로드 포함)
- Internet 필요 X (BGE-M3 dataset 만 attach 되어 있으면)

## 통과 후

이 노트북이 끝까지 돌면 → phase 6 풀셋도 동일한 환경에서 돌 가능성 매우 높음. phase 6 노트북 commit 해서 ~50분 실행 → submit.

## ✅ 데이터셋 attach

- Competition data (자동)
- `BAAI/bge-m3` (스모크엔 로드만 — 실제 encode 안 함, attach 안 돼있어도 OK 표시만 다름)


In [ ]:
# Cell 1 — paths
import re, csv, json, sys, time
from pathlib import Path
from collections import defaultdict, Counter

DATA = None
for cand in [
    Path('/kaggle/input/llm-agentic-legal-information-retrieval'),
    Path('/kaggle/input/competitions/llm-agentic-legal-information-retrieval'),
]:
    if cand.exists() and (cand / 'laws_de.csv').exists():
        DATA = cand; break
if DATA is None:
    raise RuntimeError(f'Data not found. /kaggle/input contents: {list(Path("/kaggle/input").iterdir())}')
WORK = Path('/kaggle/working')
CACHE = WORK / 'cache_smoke'
CACHE.mkdir(parents=True, exist_ok=True)
print(f'DATA: {DATA}')
print(f'WORK: {WORK}')
for p in ['train.csv', 'val.csv', 'test.csv', 'laws_de.csv', 'court_considerations.csv', 'sample_submission.csv']:
    fp = DATA / p
    if fp.exists():
        print(f'  ok  {p:35s} ({fp.stat().st_size/1e6:.1f} MB)')
    else:
        print(f'  XX  {p:35s} MISSING')


In [ ]:
# Cell 2 — normalize + corpus index (fast — same as phase 6 cell 3)
import pandas as pd

ART_PATTERN = re.compile(
    r'^Art\.\s*(?P<art>\d+[a-z]?(?:bis|ter|quater)?)'
    r'(?:\s*Abs\.\s*(?P<abs>\d+[a-z]?(?:bis|ter)?))?'
    r'(?:\s*lit\.\s*(?P<lit>[a-zA-Z]+))?'
    r'(?:\s*Ziff\.\s*(?P<ziff>\d+))?'
    r'\s+(?P<code>[A-Za-z\u00c4\u00d6\u00dc\u00e4\u00f6\u00fc\u00df0-9\.\-]+)$'
)

def parse_article(c):
    m = ART_PATTERN.match(c.strip()); return m.groupdict() if m else None

def normalize_whitespace(c):
    s = re.sub(r'\s+', ' ', c.strip())
    s = re.sub(r'\bArt\.(?=\d)', 'Art. ', s)
    s = re.sub(r'\bAbs\.(?=\d)', 'Abs. ', s)
    s = re.sub(r'\blit\.(?=[a-zA-Z])', 'lit. ', s)
    return s

print('Loading laws + court citation sets ...')
t0 = time.time()
laws_df = pd.read_csv(DATA / 'laws_de.csv')
laws_cits = set(laws_df['citation'].astype(str))
court_cits = set()
with open(DATA / 'court_considerations.csv', encoding='utf-8') as f:
    r = csv.reader(f); next(r, None)
    for row in r:
        if row: court_cits.add(row[0])
CORPUS = laws_cits | court_cits
print(f'  {len(CORPUS):,} citations in {time.time()-t0:.1f}s')

PARENT_TO_CHILDREN = defaultdict(list)
for c in laws_cits:
    m = ART_PATTERN.match(c)
    if not m: continue
    parent = f"Art. {m.group('art')} {m.group('code')}"
    if c != parent:
        PARENT_TO_CHILDREN[parent].append(c)

def resolve_against_corpus(cit):
    c = normalize_whitespace(cit)
    if c in CORPUS: return [c]
    if c in PARENT_TO_CHILDREN: return list(PARENT_TO_CHILDREN[c])
    p = parse_article(c)
    if p:
        parent = f"Art. {p['art']} {p['code']}"
        if parent != c and parent in CORPUS: return [parent]
    return []

def granularity_filter(cits):
    cs = set(cits); keep=[]
    for c in cits:
        kids = PARENT_TO_CHILDREN.get(c)
        if kids and any(k in cs for k in kids): continue
        keep.append(c)
    return keep

def dedup(cits):
    seen=set(); out=[]
    for c in cits:
        if c not in seen: seen.add(c); out.append(c)
    return out

def split_cits(s):
    if not isinstance(s, str) or not s.strip(): return []
    return [t.strip() for t in s.split(';') if t.strip()]

train_df = pd.read_csv(DATA / 'train.csv')
total=resolved=0
for _, r in train_df.iterrows():
    for g in split_cits(r['gold_citations']):
        total += 1
        if resolve_against_corpus(g): resolved += 1
print(f'train resolve: {resolved}/{total} = {resolved/max(1,total):.4f}')
assert resolved/max(1,total) > 0.95, 'normalization regression'
print('  ✅ normalize OK')


In [ ]:
# Cell 3 — verify BGE-M3 loads (no encoding yet — just checks the path)
BGE_M3_LOADED = False
try:
    from sentence_transformers import SentenceTransformer
    for p in [
        '/kaggle/input/bge-m3/bge-m3',
        '/kaggle/input/bge-m3',
        '/kaggle/input/baai-bge-m3',
        'BAAI/bge-m3',
    ]:
        try:
            print(f'  trying {p} ...')
            _model = SentenceTransformer(p)
            print(f'  ✅ BGE-M3 loaded from {p}')
            # encode a single dummy to verify it actually runs
            _ = _model.encode(['Smoke test query'], normalize_embeddings=True)
            print(f'  ✅ BGE-M3 encode test OK')
            BGE_M3_LOADED = True
            del _model
            break
        except Exception as e:
            print(f'  failed: {type(e).__name__}: {str(e)[:80]}')
except ImportError:
    print('  sentence_transformers not installed — pip install sentence-transformers')

if not BGE_M3_LOADED:
    print('  ⚠️ BGE-M3 not loadable — phase 6 dense retrieval will be unavailable')
    print('  → attach BAAI/bge-m3 dataset or enable Internet ON')


In [ ]:
# Cell 4 — BM25 on laws_de (no dense — keeps smoke fast)
!pip install -q rank_bm25
import pickle
from rank_bm25 import BM25Okapi

TOKEN_RE = re.compile(r'[A-Za-z\u00c4\u00d6\u00dc\u00e4\u00f6\u00fc\u00df0-9]+')
def tokenize(text):
    if not isinstance(text, str): return []
    return [t.lower() for t in TOKEN_RE.findall(text)]

laws_citations = laws_df['citation'].astype(str).tolist()
laws_texts = laws_df['text'].fillna('').astype(str).tolist()

bm25_path = CACHE / 'bm25.pkl'
if bm25_path.exists():
    print('Loading cached BM25 ...')
    with open(bm25_path, 'rb') as f: bm25 = pickle.load(f)
else:
    print(f'Tokenizing {len(laws_texts):,} laws texts ...')
    t0 = time.time()
    tokenized = [tokenize(t) for t in laws_texts]
    bm25 = BM25Okapi(tokenized)
    print(f'  built BM25 in {time.time()-t0:.1f}s')
    with open(bm25_path, 'wb') as f: pickle.dump(bm25, f)
print(f'  ✅ BM25 ready: {len(laws_citations):,} docs')


In [ ]:
# Cell 5 — topic classifier + minimal anchors (smoke version, subset of phase 6)
UNIVERSAL_DEFAULTS = [
    'Art. 100 Abs. 1 BGG', 'Art. 42 Abs. 1 BGG', 'Art. 42 Abs. 2 BGG',
    'Art. 95 BGG', 'Art. 106 Abs. 2 BGG',
]

TOPIC_DEFAULTS = {
    'family':     ['Art. 125 Abs. 1 ZGB', 'Art. 133 Abs. 1 ZGB', 'Art. 176 Abs. 1 ZGB',
                   'Art. 277 Abs. 1 ZGB', 'Art. 285 Abs. 1 ZGB', 'Art. 308 Abs. 1 ZGB'],
    'erbrecht':   ['Art. 457 Abs. 1 ZGB', 'Art. 467 ZGB', 'Art. 469 Abs. 1 ZGB',
                   'Art. 471 ZGB', 'Art. 505 Abs. 1 ZGB'],
    'detention':  ['Art. 221 Abs. 1 StPO', 'Art. 222 StPO', 'Art. 226 Abs. 1 StPO',
                   'Art. 382 Abs. 1 StPO', 'Art. 393 Abs. 1 StPO', 'Art. 428 Abs. 1 StPO'],
    'strafrecht': ['Art. 29 Abs. 2 BV', 'Art. 32 Abs. 1 BV', 'Art. 1 StGB',
                   'Art. 12 Abs. 1 StGB', 'Art. 47 Abs. 1 StGB'],
    'mietrecht':  ['Art. 257 OR', 'Art. 257d Abs. 1 OR', 'Art. 271 Abs. 1 OR'],
    'sachenrecht':['Art. 641 Abs. 1 ZGB', 'Art. 656 Abs. 1 ZGB', 'Art. 963 Abs. 1 ZGB'],
}

KW = {
    'family':     ['divorce', 'marriage', 'spouse', 'custody of child', 'custody of children',
                   'maintenance for', 'ehegatt', 'scheidung', 'unterhalt'],
    'erbrecht':   ['inheritance', 'estate', 'heir', 'will of', 'testament', 'erbschaft'],
    'detention':  ['detention', 'pre-trial', 'pretrial', 'remand', 'untersuchungshaft',
                   'risk of collusion', 'risk of recidivism'],
    'mietrecht':  ['lease', 'rental', 'tenancy', 'tenant', 'landlord', 'mietvertrag'],
    'sachenrecht':['ownership', 'real estate', 'eigentum', 'grundbuch'],
    'strafrecht': ['criminal', 'theft', 'fraud', 'assault', 'strafrecht'],
}
PRIORITY = ['family','erbrecht','mietrecht','sachenrecht','detention','strafrecht']

def classify(query):
    q = query.lower()
    for t in PRIORITY:
        if any(kw in q for kw in KW[t]): return t
    return 'default'

def get_defaults(topic):
    out = list(UNIVERSAL_DEFAULTS)
    if topic in TOPIC_DEFAULTS: out.extend(TOPIC_DEFAULTS[topic])
    return out

val_df = pd.read_csv(DATA / 'val.csv')
print('val topic classification:')
for _, r in val_df.iterrows():
    t = classify(r['query'])
    print(f"  {r['query_id']}: {t:12s}  (defaults: {len(get_defaults(t))})")


In [ ]:
# Cell 6 — predict (smoke = seeds + anchors + BM25 only, no dense/court/rerank/LLM)
import numpy as np

RE_ART_Q  = re.compile(r'\bArt(?:icle)?\.?\s*(\d+[a-z]?(?:bis|ter|quater)?)'
                       r'(?:\s*(?:Abs\.|al\.|para\.)\s*(\d+))?'
                       r'(?:\s*(?:lit\.|let\.)\s*([a-z]))?'
                       r'\s+([A-Z][A-Za-z\u00c4\u00d6\u00dc\u00e4\u00f6\u00fc\u00df0-9\.\-]+)')
RE_BGE_Q  = re.compile(r'\bBGE\s+(\d+)\s+([IVX]+)\s+(\d+)(?:\s+E\.?\s*([\d\.]+))?')
RE_BGER_Q = re.compile(r'\b(\d+[A-Za-z]_\d+/\d+)(?:\s+E\.?\s*([\d\.]+))?')

def extract_seeds(query):
    seeds = []
    for m in RE_ART_Q.finditer(query):
        art, abs_, lit, code = m.groups()
        parts = [f'Art. {art}']
        if abs_: parts.append(f'Abs. {abs_}')
        if lit:  parts.append(f'lit. {lit}')
        parts.append(code)
        seeds.extend(resolve_against_corpus(' '.join(parts)))
    for m in RE_BGE_Q.finditer(query):
        vol, book, page, e = m.groups()
        base = f'BGE {vol} {book} {page}'
        if e: seeds.extend(resolve_against_corpus(f'{base} E. {e}'))
        seeds.extend(resolve_against_corpus(base))
    for m in RE_BGER_Q.finditer(query):
        case, e = m.groups()
        if e: seeds.extend(resolve_against_corpus(f'{case} E. {e}'))
        seeds.extend(resolve_against_corpus(case))
    return dedup(seeds)

TOP_N_EMIT = 22

def predict(query):
    topic = classify(query)
    seeds = extract_seeds(query)
    anchors = []
    for a in get_defaults(topic):
        r = resolve_against_corpus(a)
        if r: anchors.append(r[0])
    anchors = dedup(anchors)

    scores = bm25.get_scores(tokenize(query))
    top_idx = np.argsort(scores)[::-1][:100]
    bm25_top = [laws_citations[i] for i in top_idx]

    front = dedup(seeds + anchors)
    rest = [c for c in bm25_top if c not in set(front)]
    out = granularity_filter(dedup(front + rest))[:TOP_N_EMIT]
    return out

# Sample run on first 3 val
print('Sample predictions:')
for _, r in val_df.head(3).iterrows():
    p = predict(r['query'])
    print(f"  {r['query_id']} ({classify(r['query']):10s})  → {len(p)} cits  e.g. {p[:3]}")
print('  ✅ predict pipeline OK')


In [ ]:
# Cell 7 — val Macro F1 (sanity, expect 0.05~0.15)
def per_query_f1(g, p):
    g, p = set(g), set(p)
    if not g and not p: return 1.0
    if not g or not p: return 0.0
    tp = len(g & p)
    if tp == 0: return 0.0
    pr = tp/len(p); rc = tp/len(g); return 2*pr*rc/(pr+rc)

def macro_f1(gd, pd_):
    qs = set(gd) | set(pd_)
    return sum(per_query_f1(gd.get(q, []), pd_.get(q, [])) for q in qs) / max(1, len(qs))

gold = {r['query_id']: split_cits(r['gold_citations']) for _, r in val_df.iterrows()}
preds = {r['query_id']: predict(r['query']) for _, r in val_df.iterrows()}
f1 = macro_f1(gold, preds)
print(f'VAL Macro F1 (smoke) = {f1:.4f}')
print('expected range 0.05~0.15 (smoke is BM25-only baseline)')
print('per-query:')
for qid in sorted(gold):
    print(f"  {qid}: F1={per_query_f1(gold[qid], preds[qid]):.3f}  gold={len(gold[qid])}  pred={len(preds[qid])}")


In [ ]:
# Cell 8 — submission.csv + corpus coverage sanity
test = pd.read_csv(DATA / 'test.csv')
rows = []
print(f'Predicting test ({len(test)} rows) ...')
t0 = time.time()
for _, r in test.iterrows():
    cits = predict(r['query'])
    rows.append({'query_id': r['query_id'], 'predicted_citations': ';'.join(cits)})
print(f'  done in {time.time()-t0:.1f}s')

sub = pd.DataFrame(rows)
out = WORK / 'submission.csv'
sub.to_csv(out, index=False)
print(f'\n✅ Wrote {out}  rows={len(sub)}')
print(sub.head(3).to_string(index=False))

all_preds = []
for cs in sub['predicted_citations']:
    all_preds.extend(cs.split(';'))
unique = set(all_preds)
in_corpus = sum(1 for c in unique if c in CORPUS)
print(f'\ncorpus coverage: {in_corpus}/{len(unique)} = {in_corpus/max(1,len(unique)):.4f}')
if in_corpus / max(1, len(unique)) < 0.99:
    print('  ⚠️ some predicted cits are NOT in corpus — investigate before phase 6')
else:
    print('  ✅ all predictions corpus-valid — phase 6 should work end-to-end')

counts = sub['predicted_citations'].str.count(';').add(1)
print(f'\nemit count: min={counts.min()} mean={counts.mean():.1f} max={counts.max()}')
print('\n=== SMOKE TEST COMPLETE ===')
print('If you got here cleanly, phase6_signals.ipynb should run too.')
